# Train BLIP Baseline

This notebook fine-tunes a BLIP VQA baseline with LoRA adapters on the processed multi-task JSONL data.

Start with a very small subset first. The pretrained BLIP weights stay frozen; only lightweight LoRA adapter weights are trained.

## 1. Colab Repository Setup

Run this cell first in Colab. It clones or updates the repository and changes the working directory to the repo root.

In [ ]:
from pathlib import Path
import os
import subprocess

REPO_URL = "https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git"
COLAB_REPO_DIR = Path("/content/multi-task-moe-vlm-assistant")

if Path("/content").exists():
    if not COLAB_REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(COLAB_REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "pull", "origin", "main"], check=True, cwd=COLAB_REPO_DIR)

    os.chdir(COLAB_REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

## 2. Setup

Run this notebook from the repository root, or let the cell below add the project root to `sys.path`.

In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys

os.environ.setdefault("CUDA_LAUNCH_BLOCKING", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

def is_project_root(candidate: Path) -> bool:
    required_paths = [
        "scripts/prepare_data.py",
        "scripts/build_multitask_data.py",
        "src/data/preprocessing.py",
        "src/models/baseline_vlm.py",
        "notebooks/train_blip_baseline.ipynb",
    ]
    return all((candidate / path).exists() for path in required_paths)


def find_project_root(start: Path) -> Path:
    candidates = [
        Path("/content/multi-task-moe-vlm-assistant"),
        Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant"),
        start,
        *start.parents,
    ]

    for candidate in candidates:
        if candidate == Path("/"):
            continue
        if is_project_root(candidate):
            return candidate

    raise FileNotFoundError(
        "Could not find the project root. In Colab, run: "
        "!git clone https://github.com/kdnehihi/multi-task-moe-vlm-assistant.git && "
        "%cd /content/multi-task-moe-vlm-assistant"
    )


PROJECT_ROOT = find_project_root(Path.cwd())

sys.path.append(str(PROJECT_ROOT))
PROJECT_ROOT

In [ ]:
try:
    import peft
except ModuleNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)

import torch
from peft import LoraConfig, get_peft_model
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from transformers import BlipForQuestionAnswering, BlipProcessor

torch.__version__

## 3. Config

Keep these values small for the first run. The default uses LoRA adapters so the pretrained BLIP weights stay frozen. If CUDA throws a device-side assert or OOM, restart the runtime before rerunning CUDA cells.

In [ ]:
METADATA_PATH = PROJECT_ROOT / "data/processed/multitask/validation.jsonl"
MODEL_NAME = "Salesforce/blip-vqa-base"
LOCAL_CHECKPOINT_DIR = PROJECT_ROOT / "outputs/checkpoints/blip_lora_baseline_sample"
DRIVE_CHECKPOINT_DIR = Path("/content/drive/MyDrive/multi-task-moe-vlm-assistant/checkpoints/blip_lora_baseline_sample")
USE_GOOGLE_DRIVE_CHECKPOINT = True

# Start small. LoRA is much lighter than full fine-tuning, but BLIP still needs careful VRAM use.
MAX_SAMPLES = 100
BATCH_SIZE = 1
EPOCHS = 1
LEARNING_RATE = 1e-4
GRADIENT_ACCUMULATION_STEPS = 4
TRAIN_MODE = "lora"
LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["query", "value"]
SEED = 42
QUESTION_MAX_LENGTH = 64
ANSWER_MAX_LENGTH = 32

DATA_LIMITS = {
    "docvqa": 1667,
    "chartqa": 1667,
    "textvqa": 1666,
}
EXPECTED_TOTAL_SAMPLES = sum(DATA_LIMITS.values())

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
USE_MIXED_PRECISION = DEVICE == "cuda"
DEVICE

## 4. Checkpoint Storage

In Colab, checkpoints should be saved to Google Drive because `/content` is temporary. If Drive mount fails, the notebook falls back to the local `outputs/checkpoints` directory.

In [ ]:
CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

if USE_GOOGLE_DRIVE_CHECKPOINT:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        CHECKPOINT_DIR = DRIVE_CHECKPOINT_DIR
    except Exception as error:
        print(f"Google Drive checkpoint storage is unavailable: {error}")
        print("Using local checkpoint directory instead. Download the checkpoint manually if needed.")
        CHECKPOINT_DIR = LOCAL_CHECKPOINT_DIR

print(f"Checkpoint directory: {CHECKPOINT_DIR}")

## 5. Prepare Data If Needed

This cell downloads raw samples and builds `data/processed/multitask/validation.jsonl` if the processed file is missing or incomplete. In Colab, this is the cell that recreates the local data instead of requiring data to be committed to GitHub.

In [ ]:
def count_jsonl_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


current_count = count_jsonl_lines(METADATA_PATH)
print(f"Current processed examples: {current_count}")

if current_count < EXPECTED_TOTAL_SAMPLES:
    for dataset_name, limit in DATA_LIMITS.items():
        command = [
            sys.executable,
            str(PROJECT_ROOT / "scripts/prepare_data.py"),
            "--dataset",
            dataset_name,
            "--split",
            "validation",
            "--limit",
            str(limit),
            "--streaming",
        ]
        print("Running:", " ".join(command))
        subprocess.run(command, check=True, cwd=PROJECT_ROOT)

    build_command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts/build_multitask_data.py"),
        "--split",
        "validation",
    ]
    print("Running:", " ".join(build_command))
    subprocess.run(build_command, check=True, cwd=PROJECT_ROOT)
else:
    print("Processed data already exists. Skipping data preparation.")

print(f"Final processed examples: {count_jsonl_lines(METADATA_PATH)}")

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

records = []
with METADATA_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))

random.shuffle(records)
records = records[:MAX_SAMPLES]

len(records), records[0]

## 6. Dataset and Collate Function

Each sample uses the first ground-truth answer as the training target. This is a simple baseline choice; later experiments can sample among acceptable answers.

In [ ]:
class BlipVQAFinetuneDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        example = self.examples[index]
        answer = example["answers"][0] if example["answers"] else ""
        return {
            "image_path": example["image_path"],
            "question": example["question"],
            "answer": answer,
            "dataset": example["dataset"],
            "task_type": example["task_type"],
        }


processor = BlipProcessor.from_pretrained(MODEL_NAME)


def collate_fn(batch):
    images = [Image.open(PROJECT_ROOT / item["image_path"]).convert("RGB") for item in batch]
    questions = [item["question"] for item in batch]
    answers = [item["answer"] for item in batch]

    inputs = processor(
        images=images,
        text=questions,
        padding=True,
        truncation=True,
        max_length=QUESTION_MAX_LENGTH,
        return_tensors="pt",
    )
    label_encoding = processor.tokenizer(
        answers,
        padding=True,
        truncation=True,
        max_length=ANSWER_MAX_LENGTH,
        return_tensors="pt",
    )
    decoder_input_ids = label_encoding.input_ids.clone()
    labels = decoder_input_ids.clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["decoder_input_ids"] = decoder_input_ids
    inputs["decoder_attention_mask"] = label_encoding.attention_mask
    inputs["labels"] = labels

    return inputs


train_dataset = BlipVQAFinetuneDataset(records)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
)

len(train_dataset)

## 7. Inspect One Batch

Run this before training. It catches invalid token ids before CUDA sees them.

In [ ]:
debug_batch = next(iter(train_loader))
vocab_size = processor.tokenizer.vocab_size

for key, value in debug_batch.items():
    print(key, value.shape, value.dtype)

valid_labels = debug_batch["labels"][debug_batch["labels"] != -100]
print("label min/max:", int(valid_labels.min()), int(valid_labels.max()))
print("decoder input min/max:", int(debug_batch["decoder_input_ids"].min()), int(debug_batch["decoder_input_ids"].max()))
print("tokenizer vocab size:", vocab_size)

assert int(valid_labels.min()) >= 0
assert int(valid_labels.max()) < vocab_size
assert int(debug_batch["decoder_input_ids"].min()) >= 0
assert int(debug_batch["decoder_input_ids"].max()) < vocab_size
print("Batch check passed.")

## 8. CUDA Health Check

Run this before loading the model. After a CUDA device-side assert, the runtime is usually poisoned and must be restarted.

In [ ]:
def check_cuda_health() -> None:
    if DEVICE != "cuda":
        print(f"CUDA health check skipped because DEVICE={DEVICE!r}.")
        return

    try:
        torch.cuda.empty_cache()
        probe = torch.randn((2, 2), device="cuda")
        _ = probe @ probe
        torch.cuda.synchronize()
        print("CUDA health check passed.")
    except Exception as error:
        raise RuntimeError(
            "CUDA is not healthy in this runtime. Restart or disconnect/delete the Colab runtime, "
            "then run the notebook again from the repository setup cell. Rerunning cells is not enough "
            "after a device-side assert."
        ) from error


check_cuda_health()

## 9. Free GPU Memory

Run this before loading the model, especially after an out-of-memory error.

In [ ]:
import gc

for name in ["model", "optimizer", "scaler"]:
    if name in globals():
        del globals()[name]

gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    print(torch.cuda.memory_summary(device=None, abbreviated=True))
else:
    print(f"GPU cleanup skipped because DEVICE={DEVICE!r}.")

## 10. Load Model

In [ ]:
check_cuda_health()
model = BlipForQuestionAnswering.from_pretrained(MODEL_NAME)
model.config.use_cache = False

def apply_lora_adapter(model):
    if TRAIN_MODE != "lora":
        raise ValueError(f"Unsupported TRAIN_MODE: {TRAIN_MODE}. This notebook is configured for LoRA fine-tuning.")

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=LORA_DROPOUT,
        bias="none",
    )
    return get_peft_model(model, lora_config)


model = apply_lora_adapter(model)

try:
    model.to(DEVICE)
except Exception as error:
    raise RuntimeError(
        f"Failed to move BLIP LoRA model to DEVICE={DEVICE!r}. If this happened after a CUDA assert, "
        "restart the runtime first. To smoke test without CUDA, temporarily set DEVICE = 'cpu'."
    ) from error

model.train()

trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(parameter.numel() for parameter in trainable_parameters)
print(f"Train mode: {TRAIN_MODE}")
print(f"LoRA target modules: {LORA_TARGET_MODULES}")
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
model.print_trainable_parameters()
trainable_params

## 11. Train

This trains only LoRA adapter weights while the pretrained BLIP base model stays frozen.

In [ ]:
if DEVICE == "cuda":
    torch.cuda.empty_cache()

autocast_device = "cuda" if DEVICE == "cuda" else "cpu"
scaler = torch.cuda.amp.GradScaler(enabled=USE_MIXED_PRECISION)
loss_history = []
optimizer.zero_grad(set_to_none=True)

for epoch in range(EPOCHS):
    for step, batch in enumerate(train_loader, start=1):
        batch = {key: value.to(DEVICE) for key, value in batch.items()}

        try:
            with torch.autocast(device_type=autocast_device, dtype=torch.float16, enabled=USE_MIXED_PRECISION):
                outputs = model(**batch)
                loss = outputs.loss
        except Exception as error:
            print(f"Failed at epoch={epoch + 1}, step={step}")
            for key, value in batch.items():
                print(key, value.shape, value.dtype, value.device)
            raise error

        loss_to_backprop = loss / GRADIENT_ACCUMULATION_STEPS
        scaler.scale(loss_to_backprop).backward()

        if step % GRADIENT_ACCUMULATION_STEPS == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        loss_value = float(loss.detach().cpu())
        loss_history.append(loss_value)

        if step % 10 == 0 or step == 1:
            print(f"epoch={epoch + 1} step={step} loss={loss_value:.4f}")

len(loss_history), loss_history[:5], loss_history[-5:]

## 12. Save Checkpoint

The checkpoint is saved under `outputs/`, which is ignored by git.

In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(CHECKPOINT_DIR)
processor.save_pretrained(CHECKPOINT_DIR)

print(f"Saved checkpoint to {CHECKPOINT_DIR}")

## 13. Quick Inference Check

Run one example through the fine-tuned checkpoint to confirm the model can generate.

In [ ]:
model.eval()

example = records[0]
image = Image.open(PROJECT_ROOT / example["image_path"]).convert("RGB")
inputs = processor(image, example["question"], return_tensors="pt")
inputs = {key: value.to(DEVICE) for key, value in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(**inputs, max_new_tokens=20)

prediction = processor.decode(output_ids[0], skip_special_tokens=True).strip()

print("question:", example["question"])
print("prediction:", prediction)
print("answers:", example["answers"])